# HR assistant

An engineering employee asks what the parental leave policy is in Germany.
The agent calls the gateway MCP server as `employee_eng`. It should search,
then cite the policy source, version, and sync time.

The same question as `outsider` is empty. The agent must say it found nothing
and must not guess a policy.

This notebook needs `OPENROUTER_API_KEY` and a model that can call tools.
`openai/gpt-4o-mini` works. A small model usually does not. Set
`OPENROUTER_MODEL`.

The gateway must be up, and the Germany policy must already be ingested.


In [ ]:
import os
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "agents_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import agents_demo

print("model", os.environ.get("OPENROUTER_MODEL", ""))


In [ ]:
QUESTION = "What is our parental leave policy in Germany?"

agent = agents_demo.build_agent("employee_eng", agents_demo.HR_ROLE)
report = agents_demo.run(agent, QUESTION)
agents_demo.print_trace(report["trace"])
print(report["answer"])

hits = []
for row in report["trace"]:
    payload = row["payload"]
    if isinstance(payload, dict):
        hits.extend(payload.get("results") or [])
matched = [hit for hit in hits if hit.get("object_id") == "pol-parental-leave-de"]
if not matched:
    raise SystemExit(f"HR agent missed the Germany policy: {hits}")
hit = matched[0]
print(
    "citation",
    hit["source_id"],
    hit["object_id"],
    "v" + str(hit["version"]),
    "synced",
    hit["synced_at"],
)
if not hit.get("version") or not hit.get("synced_at"):
    raise SystemExit(f"citation is incomplete: {hit}")
if "pol-parental-leave-de" not in report["answer"]:
    raise SystemExit(f"answer did not cite the policy: {report['answer']}")


In [ ]:
denied_agent = agents_demo.build_agent("outsider", agents_demo.HR_ROLE)
denied = agents_demo.run(denied_agent, QUESTION)
agents_demo.print_trace(denied["trace"])
print(denied["answer"])

searches = [row for row in denied["trace"] if row["tool"] == "search_knowledge"]
if not searches or any((row["result_count"] or 0) > 0 for row in searches):
    raise SystemExit(f"outsider should see no policy: {denied['trace']}")
if "pol-parental-leave-de" in denied["answer"]:
    raise SystemExit(denied["answer"])
print("outsider saw nothing")
